# Case study: steel EWS (v1 production path)

Walks the shipped pipeline interactively: TimesFM 2.5 zero-shot bands, ACI online
calibration, breach flags with changepoint/GARCH cross-checks, and candidate-surfacing
attribution. The batch equivalent is `python -m src.pipeline --config config/steel.yaml`;
the validated results live in `outputs/step*_summary.json` and the README.

Run from the repo root kernel (`pip install -e .` first).

In [ ]:
import sys, yaml
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
load_dotenv()
cfg = yaml.safe_load(open(ROOT / 'config' / 'steel.yaml'))
cfg['vertical'], cfg['calibration']['method']

In [ ]:
# Ingest: prices (macro optional without FRED key) + Federal Register events
from src.data import prices as prices_mod
from src.data import events as events_mod

panel = prices_mod.build_price_panel(cfg)
events = events_mod.build_event_table(cfg)
target = panel[cfg['target']['symbol']].dropna()
len(target), len(events)

In [ ]:
# TimesFM 2.5 zero-shot quantile forecast for the most recent block
from src.forecast.timesfm_model import TimesFMForecaster

H = cfg['forecast']['horizon_days']
fc = TimesFMForecaster(max_context=cfg['forecast']['context_days']).forecast(
    target.iloc[:-H], H, cfg['forecast']['quantiles'])
fc.q(0.1).to_frame('q10').join(fc.q(0.5).rename('q50')).join(fc.q(0.9).rename('q90'))

In [ ]:
# Full production run: cached band history -> ACI online -> filtered alerts
from src.pipeline import run

alerts = run(cfg)
print(f'{len(alerts)} alert(s) in the most recent block')
for a in alerts:
    print(a.summary())

**Interpretation guide** (from the validated studies, see README + design doc Build Log):

- ACI holds ~90% coverage through regimes (fixed offsets drift: 80% on the frozen holdout).
- Alerts require changepoint agreement by default (F1 filter: noise-to-signal 0.84 -> 0.08,
  recall preserved on the Feb-2025 shock case study).
- Attribution is candidate-surfacing with a source link, NOT a causal claim: the
  breach<->event association failed two pre-registered permutation tests
  (FR-dated p=0.46; TPU announcement-dated p=0.85).